# American College Insights

**Data Source:** [Kaggle - College Tuition, Diversity, and Pay](https://www.kaggle.com/datasets/jessemostipak/college-tuition-diversity-and-pay)

In [ ]:
import pandas as pd

historical_tuition = 'archive (1)/historical_tuition.csv'
historical_tuition_df = pd.read_csv(historical_tuition)
historical_tuition_df.head()

In [ ]:
# Constant tuition accounts for inflation, current does not
historical_tuition_df = historical_tuition_df[
    ~historical_tuition_df["tuition_type"].str.contains("Current", na=False)
]

# group 2 year, 4 year, and all types together 
grouped_hist = historical_tuition_df.groupby("tuition_type")

two_year = grouped_hist.get_group("2 Year Constant")
four_year = grouped_hist.get_group("4 Year Constant")
all_const = grouped_hist.get_group("All Constant")

# group public, private, and all together
grouped_type = historical_tuition_df.groupby("type")

private = grouped_type.get_group("Private")
public = grouped_type.get_group("Public")
all_inst = grouped_type.get_group("All Institutions")

In [ ]:
tuition_income = 'archive (1)/tuition_income.csv'
tuition_income_df = pd.read_csv(tuition_income)
tuition_income_df.head()

In [ ]:
# Important:
# Universities are pretty redundant since there are A LOT. Combine them all by state
# Income level is also very important. Same with Net cost/income level/total price
# Also compare yearly data (possible)

on_campus = tuition_income_df[tuition_income_df["campus"] == "On Campus"]
off_campus = tuition_income_df[tuition_income_df["campus"] == "Off Campus"]

low_income = tuition_income_df[
    tuition_income_df["income_lvl"].isin([
        "0 to 30,000",
        "30,001 to 48,000"
    ])
]
middle_income = tuition_income_df[tuition_income_df["income_lvl"] == "48_001 to 75,000"]
high_income = tuition_income_df[tuition_income_df["income_lvl"] == "75,001 to 110,000"]

state_map = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas", "CA": "California",
    "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware", "FL": "Florida", "GA": "Georgia",
    "HI": "Hawaii", "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa",
    "KS": "Kansas", "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi", "MO": "Missouri",
    "MT": "Montana", "NE": "Nebraska", "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey",
    "NM": "New Mexico", "NY": "New York", "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginia", "WA": "Washington", "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming",
    "DC": "District of Columbia"
}

state_df = (
    tuition_income_df
    .groupby("state")
    .agg({
        "net_cost": "mean",
        "total_price": "mean"
    })
    .reset_index()
)

state_df["state_name"] = state_df["state"].map(state_map)

In [ ]:
salary_potential = 'archive (1)/salary_potential.csv'
salary_potential_df = pd.read_csv(salary_potential)
salary_potential_df.head()

# Alignment fix: Assigning to 'df' so downstream analysis works seamlessly
df = salary_potential_df

# In the context of university outcomes and college scorecards, make_world_better_percent (or "High Meaning %") 
# is a specific metric that measures the percentage of alumni who believe their work makes the world a better place.

# Since we compared by state prior, we will do the same thing here
# We will analyze early vs mid career pay
# We will analyze make world better vs stem percent
# stem vs non-stem pay

state_df["state_name"] = state_df["state"].map(state_map)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Visualizations

In [ ]:
# Tuition over time
grouped = historical_tuition_df.groupby(["year", "tuition_type"])["tuition_cost"].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.lineplot(
    data=grouped,
    x="year",
    y="tuition_cost",
    hue="tuition_type"
)
plt.title("Tuition Cost Over Time by Type")
plt.xlabel("Year")
plt.ylabel("Average Tuition Cost")
plt.show()

In [ ]:
# Institution types and cost over time
grouped = historical_tuition_df.groupby(["year", "type"])["tuition_cost"].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.lineplot(
    data=grouped,
    x="year",
    y="tuition_cost",
    hue="type"
)
plt.title("Tuition Cost Over Time by Institution Type")
plt.xlabel("Year")
plt.ylabel("Average Tuition Cost")
plt.show()

In [ ]:
# Total price vs state
state_df_grouped = tuition_income_df.groupby("state")["total_price"].mean().sort_values()

plt.figure(figsize=(10, 12))
plt.barh(state_df_grouped.index, state_df_grouped.values)
plt.title("Average Total Price by State")
plt.xlabel("Average Total Price")
plt.ylabel("State")
plt.show()

In [ ]:
# Campus vs off-campus cost
grouped = tuition_income_df.groupby(["year", "campus"])["total_price"].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.lineplot(
    data=grouped,
    x="year",
    y="total_price",
    hue="campus"
)
plt.title("Total Price Over Time: On Campus vs Off Campus")
plt.xlabel("Year")
plt.ylabel("Average Total Price")
plt.show()

In [ ]:
# Income level vs net cost
grouped = tuition_income_df.groupby(["income_lvl", "campus"])["net_cost"].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(
    data=grouped,
    x="income_lvl",
    y="net_cost",
    hue="campus"
)
plt.title("Net Cost vs Income Level (On vs Off Campus)")
plt.xlabel("Income Level")
plt.ylabel("Average Net Cost")
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Early vs mid-career pay
df["growth"] = df["mid_career_pay"].astype(float) - df["early_career_pay"].astype(float)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="early_career_pay",
    y="mid_career_pay"
)
plt.title("Early vs Mid-Career Pay")
plt.xlabel("Early Career Pay")
plt.ylabel("Mid Career Pay")
plt.show()

plt.figure(figsize=(8, 6))
sns.histplot(df["growth"], bins=30)
plt.title("Distribution of Salary Growth (Mid - Early Career)")
plt.xlabel("Salary Growth")
plt.ylabel("Count")
plt.show()

In [ ]:
state_growth = df.groupby("state_name")["growth"].mean().sort_values()

plt.figure(figsize=(10, 12))
plt.barh(state_growth.index, state_growth.values)
plt.title("Average Salary Growth by State")
plt.xlabel("Growth (Mid - Early Career)")
plt.ylabel("State")
plt.show()

In [ ]:
# Make world better vs stem
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="stem_percent", y="make_world_better_percent")
plt.title("STEM Percent vs Make World Better Percent")
plt.show()

# STEM vs Non-Stem
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x="stem_percent", y="mid_career_pay")
plt.title("STEM Percent vs Mid Career Pay")
plt.show()